# CNC-ECM SLM: Distillation → LoRA Fine-Tuning → 8-bit GGUF → Ollama

**Goal.** Train a small, on-device language model to extract Engineering Change Management (ECM)
traceability records (entities, change events, traceability links) from bilingual (Traditional
Chinese + English) legacy CNC work-order annotations, then quantize and deploy it locally via Ollama.

**Data (real corpus).** Built from `merged_work_order_specs.csv` → `raw_annotations.jsonl`:
**217** work orders with non-empty change annotations (the `其他注意事項` field), of which **125**
are truncated at exactly 255 characters by the Microsoft Access Text-field limit. Annotations mix
Traditional Chinese and English (e.g. model codes `LA43→LA38`, `MAGNESCALE`, `KENT USA`, `UL`/`CSA`),
and contain spec revisions, quantity adjustments, and cross-order references.

**Pipeline.**
1. **Setup** — install Unsloth + dependencies on Colab Pro (A100/L4).
2. **Distillation** — a frontier *teacher* model (Paper 1 pipeline) extracts schema-valid JSON from
   each annotation → **silver** training labels. A human-annotated **gold** set is held out.
3. **Data formatting** — build `(system schema, user annotation, assistant JSON)` instruction examples.
4. **LoRA fine-tuning** — Unsloth; thesis hyperparameters adjusted for structured extraction.
5. **Quantization & export** — merge adapters → GGUF → 8-bit (Q8_0) → Ollama Modelfile.
6. **Inference with a Pydantic safety net** — validate every output; log schema-violation rate.

> **Two separations that matter.**
> (a) The *teacher* (distillation) and the *judge* (evaluation, companion paper) must be **different
> frontier models** to avoid self-preference bias.
> (b) The **gold** human-annotated records are used only for testing / judge-validation — never for
> training. Right now every row in `raw_annotations.jsonl` is `split="train"`; use the helper in §3b
> to promote a held-out gold subset.


## 0. Runtime check

Confirm you have a GPU. On Colab Pro choose **Runtime → Change runtime type → A100** (or L4).

In [ ]:
!nvidia-smi

## 1. Setup

Unsloth gives ~2x faster LoRA training and lower memory. The pinned versions below are a known-good
combination as of mid-2026; adjust if Colab's base image has moved on.

In [ ]:
# Unsloth + training stack.
# Let Unsloth resolve a COMPATIBLE trl/transformers/peft trio (version drift is the #1 cause of
# trainer hangs). Do NOT hard-pin trl unless you know it matches your transformers.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade unsloth unsloth_zoo
!pip install -q pydantic>=2 anthropic   # pydantic = safety net; anthropic = teacher API (swap provider)

# If a later cell complains about trl/transformers versions, restart the runtime
# (Runtime → Restart) and run this cell once more BEFORE anything else.
print("setup done — if you just installed, Runtime → Restart runtime, then continue from §2.")


In [ ]:
import os, json, re, random
from pathlib import Path
random.seed(42)

WORK = Path("/content/cnc_ecm")
WORK.mkdir(exist_ok=True)
(RAW := WORK / "raw").mkdir(exist_ok=True)
(OUT := WORK / "out").mkdir(exist_ok=True)
print("workspace:", WORK)


## 2. The extraction schema (Pydantic v2)

This is the same `WorkOrderECMRecord` schema from Paper 1, restated here so (a) the teacher is
prompted with it, (b) the SLM is trained to emit it, and (c) the safety net validates against it.
Trim/extend the enums to match your actual schema file.

In [ ]:
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field

class EntityType(str, Enum):
    component = "Component"
    specification = "Specification"
    regulation = "Regulation"
    exception_ticket = "ExceptionTicket"
    responsible_party = "ResponsibleParty"

class EventType(str, Enum):
    substitution = "Substitution"
    spec_revision = "SpecificationRevision"
    quantity_adjustment = "QuantityAdjustment"
    status_transition = "StatusTransition"

class LinkType(str, Enum):
    cross_order = "CrossOrderReference"
    exception_link = "ExceptionTicketLink"

class ECMEntity(BaseModel):
    entity_type: EntityType
    value: str
    is_truncated: bool = False

class ChangeEvent(BaseModel):
    event_type: EventType
    entities: List[str] = Field(default_factory=list)
    from_value: Optional[str] = None
    to_value: Optional[str] = None
    reason: Optional[str] = None
    is_truncated: bool = False

class TraceabilityLink(BaseModel):
    link_type: LinkType
    source_id: Optional[str] = None
    target_id: str

class WorkOrderECMRecord(BaseModel):
    entities: List[ECMEntity] = Field(default_factory=list)
    change_events: List[ChangeEvent] = Field(default_factory=list)
    traceability_links: List[TraceabilityLink] = Field(default_factory=list)

SCHEMA_JSON = json.dumps(WorkOrderECMRecord.model_json_schema(), ensure_ascii=False, indent=2)
print(SCHEMA_JSON[:800], "...")


## 3. Load raw annotations (real data)

Upload **`raw_annotations.jsonl`** (the 217-record file generated from your CSV) to
`/content/cnc_ecm/raw/`. Each line looks like:

```json
{"wo_id": "G961", "annotation": "2025/2/17 新增事項：\n1.尾座心軸移動方式：手輪 ...",
 "is_truncated_255": true, "raw_char_len": 255, "rev": 1, "fill_date": "...",
 "mod_date": "...", "customer_short": "KENT USA", "model": "LA34X160\"",
 "split": "train", "gold": null}
```

Use the Colab file browser (📁 left sidebar) or run the upload cell below. No synthetic fallback —
this notebook now expects your real file.

In [ ]:
raw_path = RAW / "raw_annotations.jsonl"

# If not present, prompt an upload (Colab) and move it into place.
if not raw_path.exists():
    try:
        from google.colab import files
        print("Select your raw_annotations.jsonl ...")
        up = files.upload()
        name = next(iter(up))
        Path(name).replace(raw_path)
    except Exception:
        raise FileNotFoundError(
            f"Put raw_annotations.jsonl at {raw_path} (drag into the Colab file browser)."
        )

rows = [json.loads(l) for l in open(raw_path, encoding="utf-8")]
train_rows = [r for r in rows if r["split"] == "train"]
gold_rows  = [r for r in rows if r["split"] == "gold_test"]

n_trunc = sum(r.get("is_truncated_255") for r in rows)
print(f"{len(rows)} records | {len(train_rows)} train | {len(gold_rows)} gold_test "
      f"| {n_trunc} truncated (255)")
assert len(rows) == 217, "expected 217 annotated work orders"

# peek at one real record
print("\nexample:", json.dumps(rows[0], ensure_ascii=False)[:240], "...")


### 3b. Designate a held-out gold set (do this once)

Everything arrives as `split="train"`. For judge-validation and SLM testing you need a **gold** set
with human-verified extractions. Two ways:

- **Quick start:** promote a stratified sample to `gold_test` now (truncated + non-truncated, with
  and without revision markers), hand-annotate them later, and paste each human JSON into `gold`.
- **Already annotated?** Skip this and just set `split="gold_test"` + fill `gold` in the file.

The cell below promotes a stratified ~15% sample and rewrites the file. Run it **once**, then
hand-fill the `gold` field for those rows before evaluation. Training only ever uses `split=="train"`.

In [ ]:
import random
random.seed(42)

def stratify_and_promote(rows, frac=0.15):
    # strata: (truncated?, has_revision_marker?)
    buckets = {}
    for i, r in enumerate(rows):
        key = (bool(r.get("is_truncated_255")), r.get("rev") is not None)
        buckets.setdefault(key, []).append(i)
    promoted = []
    for key, idxs in buckets.items():
        k = max(1, round(len(idxs) * frac))
        for i in random.sample(idxs, k):
            promoted.append(i)
    return set(promoted)

# Only promote if no gold set exists yet
if not any(r["split"] == "gold_test" for r in rows):
    gold_idx = stratify_and_promote(rows, frac=0.15)
    for i in gold_idx:
        rows[i]["split"] = "gold_test"
    with open(raw_path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"promoted {len(gold_idx)} records to gold_test (stratified). "
          f"Hand-fill their 'gold' field before §8.")
else:
    print("gold_test set already exists — leaving file unchanged.")

# refresh splits
train_rows = [r for r in rows if r["split"] == "train"]
gold_rows  = [r for r in rows if r["split"] == "gold_test"]
print(f"now: {len(train_rows)} train | {len(gold_rows)} gold_test")


## 4. Distillation: frontier teacher → silver JSON labels

This is the step you flagged — your raw annotations still need extracting. We call the **teacher**
(here Anthropic Claude; swap the client for whatever Paper 1 used) with the schema and a few-shot
prompt, enforce JSON, and validate with Pydantic. Invalid teacher outputs are dropped (logged), so
the training set contains only schema-valid silver labels.

Set your key first: **Colab → 🔑 Secrets → `ANTHROPIC_API_KEY`**, or `os.environ` it below.

In [ ]:
from anthropic import Anthropic
from google.colab import userdata
import re, json

# Key from Colab Secrets (🔑 left sidebar -> add ANTHROPIC_API_KEY, enable notebook access)
client = Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
TEACHER_MODEL = "claude-sonnet-4-6"   # teacher ≠ judge. Use a DIFFERENT model as judge later.

SYSTEM = (
    "You extract Engineering Change Management traceability information from a single CNC "
    "work-order annotation and return ONLY a JSON object conforming to the schema. Annotations "
    "may mix Traditional Chinese and English, contain domain abbreviations, and be truncated at "
    "255 characters mid-sentence. If an entity/event appears cut off, set its is_truncated to true. "
    "Do not invent regulation identifiers that are not present in the text.\n\nSCHEMA:\n" + SCHEMA_JSON
)

def teacher_extract(annotation: str):
    # No assistant prefill — some models reject it ("conversation must end with a user message").
    # Instruct JSON-only and extract the outermost {...} from the reply.
    msg = client.messages.create(
        model=TEACHER_MODEL, max_tokens=1500, temperature=0,
        system=SYSTEM,
        messages=[{"role": "user",
                   "content": annotation +
                   "\n\nReturn ONLY the JSON object, no prose, no markdown fences."}],
    )
    text = msg.content[0].text.strip()
    text = re.sub(r"^```json|^```|```$", "", text).strip()
    m = re.search(r"\{.*\}", text, re.DOTALL)        # grab outermost {...}
    if m:
        text = m.group(0)
    try:
        rec = WorkOrderECMRecord.model_validate_json(text)   # validate!
        return json.loads(rec.model_dump_json())
    except Exception as e:
        print("  [drop] invalid teacher output:", str(e)[:80])
        return None


In [ ]:
silver_path = OUT / "silver.jsonl"
n_ok = n_drop = 0
drops = []
with open(silver_path, "w", encoding="utf-8") as f:
    for r in train_rows:                       # NOTE: train rows only — gold is never distilled
        # give the teacher the known truncation flag as a hint (it still decides per-entity)
        hint = "\n\n[NOTE: this annotation is truncated at 255 characters.]" if r.get("is_truncated_255") else ""
        rec = teacher_extract(r["annotation"] + hint)
        if rec is None:
            n_drop += 1; drops.append(r["wo_id"]); continue
        f.write(json.dumps({
            "wo_id": r["wo_id"],
            "annotation": r["annotation"],      # store WITHOUT the hint — model trains on raw text
            "extraction": rec,
            "is_truncated_255": bool(r.get("is_truncated_255")),
            "provenance": "silver",
        }, ensure_ascii=False) + "\n")
        n_ok += 1

print(f"silver labels: {n_ok} kept, {n_drop} dropped (teacher schema-violations)")
if drops:
    print("dropped wo_ids:", drops[:20], ("..." if len(drops) > 20 else ""))


## 5. Format instruction examples

Each example is a chat with a **canonicalized** JSON target (sorted keys, compact) so the small
model learns one stable output format. We keep the schema in the system turn at train time so the
model is conditioned exactly as it will be at inference.

In [ ]:
def canon(rec: dict) -> str:
    # canonical form: drops nulls deterministically, sorts keys, compact separators
    clean = WorkOrderECMRecord.model_validate(rec).model_dump(exclude_none=True)
    return json.dumps(clean, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def to_chat(annotation: str, rec: dict):
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": annotation},
        {"role": "assistant", "content": canon(rec)},
    ]}

silver = [json.loads(l) for l in open(silver_path, encoding="utf-8")]
chat_examples = [to_chat(s["annotation"], s["extraction"]) for s in silver]
random.shuffle(chat_examples)
print(f"{len(chat_examples)} training chats")
print(json.dumps(chat_examples[0], ensure_ascii=False, indent=2)[:600])


## 6. Load the base model (Unsloth)

**Recommended: `unsloth/Qwen2.5-3B-Instruct`.** It is a plain text transformer with mature Unsloth
support, stronger Traditional Chinese pretraining (better for this bilingual corpus), and clean GGUF
export for Ollama.

`unsloth/gemma-3n-E4B-it` (thesis parity) is **multimodal** (MatFormer + Per-Layer-Embedding + audio
tower). As of mid-2026 its Unsloth support lives in `unsloth_zoo/temporary_patches` and can fail under
`torch.compile` with `'Gemma3nRMSNorm' object has no attribute 'weight'`. The cell below disables
compile as a workaround, but Qwen2.5-3B is the more reliable choice for a working end-to-end run.

> **First run downloads several GB** (the base model). It can sit with no visible progress for a few
> minutes — this is normal, not a hang. Wait for `LoRA attached to ...` before running §7.

In [ ]:
import os, torch
# --- Gemma-3n workaround: disable torch.compile (harmless for other models too) ---
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
try:
    torch.compiler.set_stance("force_eager")
except Exception:
    pass

from unsloth import FastLanguageModel

# Recommended for this project (reliable + strong Traditional Chinese):
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
# Thesis-parity alternative (multimodal; may need the compile-disable above, fragile GGUF export):
# BASE_MODEL = "unsloth/gemma-3n-E4B-it"

MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ,
    dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0.0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)
print("LoRA attached to", BASE_MODEL)


### ⚠️ Before training: run cells in order & verify prerequisites

`trainer.train()` hanging is almost always one of these:
1. **Cells §5 and §6 weren't run** — the trainer needs `chat_examples` (from §5) and
   `model`/`tokenizer`/`MAX_SEQ` (from §6). Run **§2 → §6 top-to-bottom** first.
2. **§6 model download looks like a hang** — `gemma-3n-E4B` is multi-GB; first load can take
   several minutes with no visible progress. That's normal; let it finish before running §7.
3. **Library version drift** — newer `trl` moved `dataset_text_field`/`max_seq_length` into
   `SFTConfig`. The training cell below handles both old and new APIs.

Run this check; it must print **ALL PREREQUISITES OK** before you run §7.

In [ ]:
# Prerequisite + environment check (fast, no training)
import importlib.metadata as im
def ver(p):
    try: return im.version(p)
    except Exception: return "NOT INSTALLED"
for p in ["unsloth","trl","transformers","peft","accelerate","datasets","bitsandbytes"]:
    print(f"{p:14s} {ver(p)}")

print("-"*40)
ok = True
for name in ["chat_examples","model","tokenizer","MAX_SEQ","SYSTEM"]:
    present = name in globals()
    print(f"{name:14s} {'OK' if present else 'MISSING — run its cell first'}")
    ok &= present
if ok:
    print(f"\nchat_examples: {len(chat_examples)} | first keys: {list(chat_examples[0].keys())}")
    import torch
    print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
    print("\nALL PREREQUISITES OK — safe to run §7")
else:
    print("\nDo NOT run §7 yet — run the missing cells above (in order) first.")


## 7. LoRA fine-tuning

Thesis baseline (eff. batch 4, LR 2e-4, AdamW-8bit, wd 0.01) — but extraction over a 3-layer schema
needs more passes than the thesis's ~1.1 epochs, so we target **3 epochs** and watch eval loss.
Hold out 10% as an eval split to catch overfitting on the small set.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer
import trl, inspect, torch

# Build the text field and DROP the raw "messages" column (stale columns can stall mapping/collation)
def render(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)}

ds = Dataset.from_list(chat_examples)
ds = ds.map(render, remove_columns=ds.column_names)          # <- keep ONLY "text"
ds = ds.train_test_split(test_size=0.1, seed=42)
print("train/eval sizes:", len(ds["train"]), len(ds["test"]))
print("sample text[:200]:\n", ds["train"][0]["text"][:200])

# eval/save arg name differs across transformers versions
from transformers import TrainingArguments
ta_params = inspect.signature(TrainingArguments.__init__).parameters
eval_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"

common = dict(
    per_device_train_batch_size=2, gradient_accumulation_steps=2,   # eff. batch 4
    warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear",
    logging_steps=1, save_strategy="epoch",
    output_dir=str(OUT / "checkpoints"), seed=42, report_to="none",
)
common[eval_key] = "epoch"

# TRL >=0.9 moved dataset_text_field & max_seq_length into SFTConfig; older TRL keeps them on SFTTrainer.
sig = inspect.signature(SFTTrainer.__init__).parameters
print("TRL", trl.__version__, "| SFTTrainer takes dataset_text_field:", "dataset_text_field" in sig)

if "dataset_text_field" in sig:
    # ---- old TRL path ----
    args = TrainingArguments(**common)
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=ds["train"], eval_dataset=ds["test"],
        dataset_text_field="text", max_seq_length=MAX_SEQ, packing=False, args=args,
    )
else:
    # ---- new TRL path: use SFTConfig ----
    from trl import SFTConfig
    args = SFTConfig(dataset_text_field="text", max_seq_length=MAX_SEQ,
                     packing=False, **common)
    trainer = SFTTrainer(
        model=model, processing_class=tokenizer,
        train_dataset=ds["train"], eval_dataset=ds["test"], args=args,
    )

trainer.train()


## 8. Quick sanity eval on the held-out gold set

The single most important early signal: **does the fine-tuned model emit schema-valid JSON?**
We report the schema-violation rate (frontier vs. SLM is a headline metric in the paper).

In [ ]:
FastLanguageModel.for_inference(model)

def slm_extract(annotation: str):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": annotation}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                           return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=1024, temperature=0.0, do_sample=False)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    text = re.sub(r"^```json|```$", "", text).strip()
    try:
        rec = WorkOrderECMRecord.model_validate_json(text)
        return json.loads(rec.model_dump_json()), text
    except Exception:
        return None, text

if not gold_rows:
    print("No gold_test rows yet — run §3b and hand-fill the 'gold' field first.")
else:
    valid = 0
    trunc_valid = trunc_total = 0
    for r in gold_rows:
        rec, raw = slm_extract(r["annotation"])
        ok = rec is not None
        valid += ok
        if r.get("is_truncated_255"):
            trunc_total += 1; trunc_valid += ok
        flag = "VALID" if ok else "INVALID"
        print(f"{r['wo_id']:>6} [{'T' if r.get('is_truncated_255') else ' '}]: {flag}")
    print(f"\nschema-valid overall: {valid}/{len(gold_rows)} "
          f"({100*valid/len(gold_rows):.0f}%)")
    if trunc_total:
        print(f"schema-valid on truncated: {trunc_valid}/{trunc_total}")
    # gold extractions present?
    have_gold = sum(r.get("gold") is not None for r in gold_rows)
    print(f"gold extractions filled: {have_gold}/{len(gold_rows)} "
          f"{'(ready for F1 scoring)' if have_gold==len(gold_rows) else '(fill the rest for full metrics)'}")


## 9. Merge + export to 8-bit GGUF

Q8_0 mirrors the thesis (fidelity over aggressive 4-bit). For a 3–4B model this lands around 3–4 GB.

In [ ]:
model.save_pretrained_gguf(
    str(OUT / "gguf_q8"), tokenizer,
    quantization_method="q8_0",
)
print("GGUF written to", OUT / "gguf_q8")
!ls -lh {OUT}/gguf_q8


## 10. Ollama Modelfile (run on the local edge device)

Copy the `.gguf` to your MacBook / shop-floor PC, save the Modelfile below next to it, then:

```bash
ollama create cnc-ecm -f Modelfile
ollama run cnc-ecm '粗車外徑後 換料 SUS304→SUS316L 依 ISO 9001 客訴 RMA-2401'
```

Baking the system prompt + schema into the Modelfile means callers just send the raw annotation.

In [ ]:
modelfile = f"""FROM ./model-q8_0.gguf
PARAMETER temperature 0
PARAMETER num_ctx 2048
SYSTEM \"\"\"{SYSTEM}\"\"\"
"""
Path(OUT / "Modelfile").write_text(modelfile, encoding="utf-8")
print(modelfile[:400], "...")
print("\nSaved:", OUT / "Modelfile")


## 11. Local inference with the Pydantic safety net

On the edge device, wrap the Ollama call so malformed JSON is caught (and counted). This is both a
production safeguard and a reportable metric. One bounded retry with a "return ONLY valid JSON"
nudge is usually enough.

In [ ]:
# Runs on the edge device where Ollama is serving `cnc-ecm`.
import requests, json, re

def ollama_extract(annotation: str, retries: int = 1):
    prompt = annotation
    for attempt in range(retries + 1):
        resp = requests.post("http://localhost:11434/api/generate",
                             json={"model": "cnc-ecm", "prompt": prompt, "stream": False,
                                   "options": {"temperature": 0}})
        text = resp.json()["response"]
        text = re.sub(r"^```json|```$", "", text.strip()).strip()
        try:
            return WorkOrderECMRecord.model_validate_json(text)   # safety net
        except Exception:
            prompt = annotation + "\n\nReturn ONLY a valid JSON object matching the schema."
    return None   # logged as a schema-violation

# rec = ollama_extract("數量由 50 調整為 80 pcs，參照前單 WO-231220-07")
# print(rec.model_dump_json(indent=2) if rec else "SCHEMA VIOLATION")


---
### Next steps into the paper
- Run §8 for **both** candidate models → report schema-violation rate + per-entity F1 vs. the frontier.
- Keep `provenance` flags on every training example for the limitations section (silver-label noise).
- The companion paper's **LLM-as-a-Judge** scores base / fine-tuned / frontier on ECM dimensions and
  validates the judge against the gold set. Use a **judge model different from the teacher** here.
